# HL-mesh Support structure

This notebook compute HL-meshes with support structure from two methods.

`hl_mesh_computation(name, tau)` builds an HL-mesh from a mesh in `data/ph_results/` whose
spheres lie in a linear sphere complex (`hyperplane_fitting`, `linear_sphere_complex`); the
plane of the complex is drawn. Then: a slider to show randomly sampled spheres of the L-mesh,
and the sphere-centres mesh coloured by its planarity (PH-CPF measure, in %).

**Viewer:** `kv.init()` starts a kayviz viewer and prints its URL; open it in a browser.

## Python paths and configurations

data path: `data/ph_results`

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

from LMesh.lmesh_serializers import lmesh_visualize
import kayviz as kv
import pymeshlab
from LMesh.L_mesh import Lmesh
from hanan.geometry import Mesh
from hanan.optimization.optimizer import Optimizer
from hanan.geometry.utils import (
    read_obj,
    read_off,
    normalize_vertices,
    compute_cone_axes,
    conical_mesh_smallest_curvature_sphere,
    compute_face_normals
)
import numpy as np
import igl

from hanan.optimization.optimizer import Optimizer
from hanan.optimization.planarity import Planarity

from LMesh.Optimization.Or_contactFree import Or_contactFree
from LMesh.Optimization.HLmesh_reg_strips import HLmesh_reg_strips
from LMesh.Optimization.HLmesh_reg_hex_faces import HLmesh_reg_hex_faces
from LMesh.Optimization.Tangential_Pos import TDPositive
from LMesh.Optimization.Tangential_Dist import Tangential_Dist
from LMesh.Optimization.ContactPointsProximity import ContactPointsProximity
from LMesh.Optimization.ContactPointsGlide import ContactPointsGlide

from hanan import glyphs


DATA     = os.path.join(PROJECT_ROOT, "data", "ph_results")
OUT_PATH = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "HL_results")

mesh_names = [
    "aspiralBdef.out_d_p",
    #"ph_input_cmcOptimized.out_d_p"
    #"ph_input_nautilus.out_d_p"
    "ph_input_nautilus_ex2.out_d_p"
    #"ph_input_nautilus_channel.out_d_p"
    #"ph_input_seashell_ex5.out_d_p"
    #"ph_input_seashell_ex6.out_d_p"
    #"ph_input_spiral_def.out_d_p"
    #"ph_input_spiral_defM.out_d_p"
    #"ph_input_testSimple.out_d_p"
]

def setup_optimizer(lmesh):
    opt = Optimizer()
    
    weights = {
            "Or_Contact": 2.0, 
            "TD": 0,
            "TDpos": 0,
            "HLmesh_Reg_Strips": 5e-3,
            "HLmesh_Reg_Hex_Faces": 5e-3, 
            "planarity": 1,
            "npf_fairness": 0    
            }

    cf  = lmesh.top.vertices[:, :3]
    rf  = lmesh.top.vertices[:, 3]
    npf = lmesh.PF[:, :3]
    hpf = lmesh.PF[:, 3]

    # npf/hpf are per-FACE variables (one oriented plane per L-mesh face),
    # so fairness must use face->face adjacency.  A vertex adjacency list
    # would index past the end of the face array.
    plane_adj = lmesh.top.face_face_adjacency_list()
    si_sj_e = lmesh.top.edge_vertices()  # edge->(start_vertex, end_vertex) pairs

    scV, scF = lmesh.sphere_centers_mesh()
    nc_f = compute_face_normals(scV, scF)



    opt.add_variable("cv", cf.flatten())
    opt.add_variable("rv", rf)
    opt.add_variable("npf", npf.flatten())
    opt.add_variable("hpf", hpf.flatten())
    opt.add_variable("nc_f", nc_f.flatten())


    # Oriented contact term
    Or_contact = Or_contactFree()
    Or_contact.name = "or_contact"
    opt.add_objective_term(
        Or_contact,
        args=(lmesh.top,),
        w=weights["Or_Contact"],
        ce=1,
    )

    # Regularity of strips
    HLmesh_Reg_Strips = HLmesh_reg_strips()
    HLmesh_Reg_Strips.name = "HLmesh_Reg_Strips"
    opt.add_objective_term(
        HLmesh_Reg_Strips,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Strips"],
        ce=1,
    )

    Tangential_Dist_term = Tangential_Dist()
    Tangential_Dist_term.name = "tangential_dist"
    opt.add_objective_term(
        Tangential_Dist_term,
        args=(si_sj_e, ),
        w=weights["TD"],
        ce=1,
    )

    Tangential_Dist_pos_term = Tangential_Dist()
    Tangential_Dist_pos_term.name = "tangential_dist_pos"
    opt.add_objective_term(
        Tangential_Dist_pos_term,
        args=(si_sj_e, ),
        w=weights["TDpos"],
        ce=1,
    )


    # Regularity of hex faces
    HLmesh_Reg_Hex_Faces = HLmesh_reg_hex_faces()
    HLmesh_Reg_Hex_Faces.name = "HLmesh_Reg_Hex_Faces"
    opt.add_objective_term(
        HLmesh_Reg_Hex_Faces,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Hex_Faces"],
        ce=1,
    )

    #Sphere centers planarity term
    CentersPlanarity_term = Planarity()
    CentersPlanarity_term.name = "centers_planarity"
    opt.add_objective_term(
        CentersPlanarity_term,
        args=(lmesh.top.faces, "cv", "nc_f"),
        w=weights["planarity"],
        ce=1,
    )


    # if Vr is not None and Fr is not None:

    #     ContactPointsProximity_term = ContactPointsProximity()
    #     ContactPointsProximity_term.name = "contactProximity"
    #     opt.add_objective_term(
    #         ContactPointsProximity_term,
    #         args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
    #         w=weights["contactProximity"],
    #         ce=1,
    #     )
            
    #     ContactPointsGlide_term = ContactPointsGlide()
    #     ContactPointsGlide_term.name = "contactGliding"
    #     opt.add_objective_term(
    #         ContactPointsGlide_term,
    #         args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
    #         w=weights["contactGliding"],
    #         ce=1,
    #     )



    # Normal vector fairness
    opt.set_fairness("npf", plane_adj, 3, 0.1, 10, weights["npf_fairness"])
    opt.unitize_variable("npf", 3, 10)
    #opt.prev_iteration_regularizer(w=1e-4)
    opt.initialize_optimizer(verbose=True, adaptive_mu=False)


    return opt

def hyperplane_fitting(pts):
    """
    Fit a hyperplane to a set of points using SVD.

    Parameters:
    pts (numpy.ndarray): An NxD array of points, where N is the number of points and D is the dimension.

    Returns:
    normal (numpy.ndarray): The normal vector of the fitted hyperplane.
    distance_from_origin (float): The distance of the hyperplane from the origin.
    """
    # Compute the centroid of the points
    centroid = np.mean(pts, axis=0)

    # Center the points by subtracting the centroid
    centered_pts = pts - centroid

    # Perform Singular Value Decomposition (SVD)
    _, _, vh = np.linalg.svd(centered_pts)

    # The normal vector of the hyperplane is given by the last row of vh
    normal = vh[-1]

    return normal, -np.dot(normal, centroid)


def linear_sphere_complex(vertices, cone_axes, cone_angles, N, H):
    """ 
    Function to compute a linear sphere complex from cone axes, angles of the input PH-mesh at each vertex, and normal and offset of the intersecting plane in R^4. The function returns the centers and radii of the spheres in the complex.  
    Input: 
        cone_axes: (N,3) array of cone axes at each vertex of the PH-mesh
        cone_angles: (N,) array of cone half-angles at each vertex of the PH-mesh
        N: (4,) array of normal vector of the intersecting plane in R^4
        H: scalar offset of the intersecting plane in R^4
    """

    # Compute the plane G in R^3 from the plane in R^4 intersection with R^3 (x,y,z,0)
    ng = N[:-1] / np.linalg.norm(N[:-1])
    hg = H / np.linalg.norm(N[:-1])

    # Compute constant ratio betwen sphere center distance to plane and radius
    Const = np.tan( np.acos(np.clip(np.dot(N, [0, 0, 0, 1]), -1.0, 1.0)))

    sin_hangles = np.sin(cone_angles)

    # Compute the radius of the spheres in the complex
    radii = sin_hangles * Const * ( np.dot(vertices, ng) + hg )/(sin_hangles - Const * np.dot(cone_axes, ng))

    scaled_axes = cone_axes / sin_hangles[:, np.newaxis]

    centers = vertices + radii[:, np.newaxis] * scaled_axes

    return centers, radii




def hl_mesh_computation(name, tau):

    exp_dir = os.path.join(OUT_PATH, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)

    print("Reading mesh:", name)


    kv.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )

    scaled_axes = axes / np.sin(angles[:, None])

    radii   = tau * smallest_radii
    centers = V + radii[:, np.newaxis] * scaled_axes

    spheres_cyclo = np.hstack((centers, radii[:, np.newaxis]))

    # Fit hyperplane in Minkowski space 
    nh, hh = hyperplane_fitting(spheres_cyclo)


    centers, radii = linear_sphere_complex(V, axes, angles, nh, hh)

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    #experiment_name = f"{name}_tau_{tau :.2f}"
    #lmesh.export_data(exp_dir, "init")

    lmesh.improve_contact()
        # state is fully rebuilt here and safe to export.
      
    lmesh_visualize(lmesh)
    n, h = nh[:3] / np.linalg.norm(nh[:3]), hh / np.linalg.norm(nh[:3])          # plane G
    kv.register_surface_mesh("Plane G", *glyphs.plane_covering(n, h, V), color=(0.1, 0.1, 0.1), opacity=0.4)

    d = centers @ n + h                         # signed distance of each centre to G (d/r is constant)
    cut = np.abs(d) < np.abs(radii)             # spheres that cut G
    C = centers[cut] - d[cut, None] * n         # circle centres on G
    R = np.sqrt(radii[cut]**2 - d[cut]**2)      # circle radii
    kv.register_curve_network("Sphere ∩ G", *glyphs.circles(C, np.tile(n, (len(C), 1)), R), color=(0.8, 0.2, 0.2))


    # # lmesh.export_data(
    # #     output_path, experiment_name )

    opt = setup_optimizer(lmesh)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
        # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)

    # # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)

    opt.print_report()
    lmesh.improve_contact()

    # lmesh.export_data(exp_dir, "lmesh")
    
    # # kv.register_surface_mesh("Gauss L-mesh", onormals, lmesh.top.dual_top(), color=(0.8, 0.8, 0.8), opacity=0.1, show_edges=True, edge_color=(1.0, 0.5, 0.1), edge_radius = 0.002)
    # lmesh_visualize(lmesh) 

    return lmesh

kv.init()   # pings the server; raises if it is not running

ph_mesh_names = [
    "aspiralBdef.out_d_p",
    #"ph_input_cmcOptimized.out_d_p",
    "ph_input_nautilus_ex2.out_d_p",
    #"ph_input_nautilus_channel.out_d_p",
    #"ph_input_seashell_ex5.out_d_p",
    #"ph_input_seashell_ex6.out_d_p",
    #"ph_input_spiral_def.out_d_p",
    #"ph_input_spiral_defM.out_d_p",
    #"TL_mesh_test",
    #"hexMeshTestSP",
    "hexMeshTest"
    #"ph_input_testSimple.out_d_p"
]


kv.clear()
lmesh = hl_mesh_computation(ph_mesh_names[1], -0.5)
kv.show()

## Sample spheres of the L-mesh

The slider shows the first *k* spheres of a random order of all the L-mesh spheres: moving it
up adds new spheres and keeps the ones already shown, moving it down removes the latest ones.
`shown` holds the indices currently displayed; **New random order** reshuffles.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

sph_centers = lmesh.sv_centers          # spheres live on the L-mesh vertices
sph_radii   = lmesh.sv_radii
n_spheres   = len(sph_radii)

rng   = np.random.default_rng()
order = rng.permutation(n_spheres)      # the random order the slider walks through
shown = order[:0]                       # indices of the spheres currently displayed

info = widgets.Label()

def show_spheres(k):
    global shown
    shown = order[:k]
    if k == 0:
        kv.remove("Sampled spheres")
    else:
        kv.register_surface_mesh("Sampled spheres",
                                 *glyphs.spheres(sph_centers[shown], sph_radii[shown], subdivisions=2),
                                 color=(0.2, 0.3, 0.9), opacity=0.5)
    info.value = f"{k} of {n_spheres} spheres" + (f"  (latest: sphere {order[k - 1]})" if k else "")

def new_order(_):
    global order
    order = rng.permutation(n_spheres)
    show_spheres(slider.value)

slider = widgets.IntSlider(value=0, min=0, max=n_spheres, step=1, description="Spheres",
                           continuous_update=False, layout=widgets.Layout(width="60%"))
slider.observe(lambda change: show_spheres(change["new"]), names="value")
reshuffle = widgets.Button(description="New random order")
reshuffle.on_click(new_order)

display(widgets.HBox([slider, reshuffle]), info)
show_spheres(slider.value)


## Sphere centers: planarity

The sphere centers, connected like the L-mesh (one vertex per sphere, one face per plane),
coloured per face by the **planarity measure for meshes** of PH-CPF (Pluta, Edelstein, Vaxman,
Ben-Chen, *Planar Hexagonal Meshing using Coordinate Power Fields*, ACM TOG 2021),
`hanan.geometry.measures.planarity_measure`: for each four consecutive vertices
$p_1 \dots p_4$ the distance between the diagonal lines $p_1p_3$ and $p_2p_4$ divided by the
average diagonal length; per face, the root mean square over its consecutive quadruplets
(0 = planar; PH-CPF reports it in percent). Pick the scalar "Planarity" on "Sphere centers".


In [ ]:
from hanan.geometry.measures import planarity_measure

scV, scF = lmesh.sphere_centers_mesh()
planarity = planarity_measure(scV, scF)

kv.register_surface_mesh("Sphere centers", scV, scF, color=(0.8, 0.8, 0.2), opacity=0.8, show_edges=True)
kv.add_scalar_quantity("Sphere centers", "Planarity", planarity, defined_on="faces")

print(f"Planarity of the {len(scF)} sphere-center faces: max {100 * planarity.max():.2f}%, "
      f"mean {100 * planarity.mean():.2f}%")
